# CoffeeLeafVision — Exploratory Data Analysis

Análisis exploratorio sobre **BRACOL + RoCoLe** combinados (~3,300 imágenes reales de hojas de café). Justifica las decisiones de modelado y limpieza implementadas en `src/preprocess.py`.

**Autor:** Juan Alvarez · Universidad de San Buenaventura

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

## 1. Cargar manifest

In [ ]:
from src.preprocess import build_manifest, save_manifest

bracol = Path("../data/BRACOL")
rocole = Path("../data/RoCoLe")
manifest = build_manifest(bracol, rocole)
save_manifest(manifest, Path("../data/manifest.csv"))
print(f"Total: {len(manifest):,} imágenes")
manifest.head()

## 2. Balance de clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
manifest["label"].value_counts().plot(kind="bar", ax=axes[0], color="#DD8452")
axes[0].set_title("Imágenes por clase de enfermedad")
axes[0].set_ylabel("Cantidad")
manifest["variety"].value_counts().plot(kind="bar", ax=axes[1], color="#4C72B0")
axes[1].set_title("Imágenes por variedad (Arabica / Robusta)")
axes[1].set_ylabel("Cantidad")
plt.tight_layout()
plt.show()

**Observaciones:**

- Distribución por clase moderadamente desbalanceada — se compensa con stratified CV + class-aware loss.
- Variedades aportan diversidad: el modelo verá tanto arábico (BRACOL) como robusta (RoCoLe).
- El split estratificado preservará proporción de clase Y variedad en cada fold.

## 3. Tamaño de imágenes

In [ ]:
sample = manifest.sample(n=min(200, len(manifest)), random_state=42)
sizes = []
for fp in sample["filepath"]:
    try:
        with Image.open(fp) as im:
            sizes.append(im.size)
    except Exception as e:
        print(f"Error reading {fp}: {e}")

sizes_df = pd.DataFrame(sizes, columns=["width", "height"])
print(sizes_df.describe().round(0))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sizes_df["width"], sizes_df["height"], alpha=0.5)
ax.set_xlabel("Ancho (px)")
ax.set_ylabel("Alto (px)")
ax.set_title("Dimensiones de las imágenes (muestra)")
plt.show()

- Las imágenes vienen en distintos tamaños. El pipeline las redimensiona a 224x224 para alimentar modelos pre-entrenados en ImageNet.
- Padding en Albumentations maneja bordes oscuros sin distorsionar la hoja.

## 4. Muestras visuales de cada clase

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for col, label in enumerate(["healthy", "leaf_rust", "leaf_miner", "phoma", "cercospora"]):
    rows = manifest[manifest["label"] == label]
    samples = rows.sample(n=min(2, len(rows)), random_state=42)
    for row_idx, (_, sample_row) in enumerate(samples.iterrows()):
        ax = axes[row_idx, col]
        with Image.open(sample_row["filepath"]) as im:
            ax.imshow(im)
        ax.axis("off")
        ax.set_title(f"{label}\n({sample_row['variety']})", fontsize=10)
plt.tight_layout()
plt.show()

**Observaciones cualitativas:**

- `healthy`: hojas verdes uniformes sin lesiones visibles.
- `leaf_rust`: pústulas amarillas-naranjas en la cara inferior — características y discriminativas.
- `leaf_miner`: surcos serpenteantes/galerías hechos por la larva.
- `phoma`: manchas necróticas circulares con halo.
- `cercospora`: manchas marrones con centro claro y halo amarillento.

Estas marcas son lo que el modelo debe aprender. Grad-CAM debería activarse sobre ellas.

## 5. Stratified split — verificación

In [ ]:
from src.preprocess import stratified_split

splits = stratified_split(manifest, test_size=0.15, n_folds=5, random_state=42)

print(f"Test set: {len(splits['test_idx'])} imágenes")
print(f"Folds: {len(splits['folds'])}")
for i, (train_idx, val_idx) in enumerate(splits['folds']):
    print(f"  Fold {i+1}: train={len(train_idx)}, val={len(val_idx)}")

In [ ]:
test_set = manifest.loc[splits["test_idx"]]
print("Distribución del hold-out test:")
print(test_set["label"].value_counts(normalize=True).round(3))
print("\nDistribución por variedad en hold-out:")
print(test_set["variety"].value_counts(normalize=True).round(3))

**Verificación:** las proporciones de clases y variedades en el hold-out son comparables a las del dataset completo. El split estratificado funcionó.

## 6. Conclusiones para modelado

- **Dataset combinado:** ~3,300 imágenes en 5 clases, dos variedades de café.
- **Desbalance moderado** — manejable con stratified CV + métricas correctas (F1 macro).
- **Imágenes heterogéneas en tamaño** — pipeline las normaliza a 224x224.
- **Las clases son visualmente distinguibles** — el problema es aprendible.
- **Próximos pasos:** `02_training.ipynb` corre en Kaggle (GPU T4 gratis) el entrenamiento de las 4 arquitecturas x 5 folds. Ver `src/train.py`.